# Ogham OCR: Unfine-tuned Baseline Evaluation

Evaluate TrOCR-small **before any fine-tuning** on the synthetic validation set.

**Purpose:** Establish that the pretrained model has zero Ogham capability,
proving that the 0.06% CER result is entirely from fine-tuning.

**Expected result:** ~95-100% CER, ~0% exact match.

**Requirements:** Colab with GPU (for faster inference on 5k images).

---
## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
!pip install -q transformers datasets huggingface_hub editdistance Pillow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ogham_ocr'
os.makedirs(f'{DRIVE_ROOT}/experiments', exist_ok=True)

In [ ]:
import os
REPO_DIR = '/content/ogham_ocr'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/newsyd04/ogham-masters.git {REPO_DIR}
%cd {REPO_DIR}
import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
from huggingface_hub import login
login()

---
## 2. Load Unfine-tuned Model

Load `trocr-small-stage1` with the Ogham tokenizer extension applied (so it can output Ogham characters) but **no training whatsoever**.

In [ ]:
import torch
from src.training.tokenizer_extension import setup_ogham_model_and_tokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load pretrained model with Ogham tokenizer extension — NO fine-tuning
model, processor, tokenizer = setup_ogham_model_and_tokenizer(
    'microsoft/trocr-small-stage1',
    init_strategy='latin',
)
model = model.to(device).eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: trocr-small-stage1 (UNFINE-TUNED)')
print(f'Parameters: {total_params:,}')
print(f'Tokenizer vocab: {len(tokenizer)}')
print(f'Device: {device}')
print(f'\nThis model has NEVER been trained on Ogham data.')
print(f'Its weights are purely from Stage 1 pretraining (synthetic English text).')

---
## 3. Load Synthetic Validation Set

In [ ]:
from datasets import load_dataset

HF_DATASET = 'DaraTraining/ogham-synthetic-200k'
print(f'Loading: {HF_DATASET} (validation split)')
val_ds = load_dataset(HF_DATASET, split='validation', token=True)
print(f'Validation samples: {len(val_ds)}')

# Show a sample
sample = val_ds[0]
print(f"\nSample 0:")
print(f"  ogham_text: {sample['ogham_text']}")
print(f"  image size: {sample['image'].size}")

---
## 4. Evaluate

In [ ]:
import editdistance
from tqdm import tqdm

def compute_cer(predictions, references):
    total_dist = 0
    total_chars = 0
    for pred, ref in zip(predictions, references):
        total_dist += editdistance.eval(pred, ref)
        total_chars += max(len(ref), 1)
    return total_dist / total_chars if total_chars > 0 else 0.0

all_preds = []
all_refs = []

print(f'Evaluating unfine-tuned TrOCR-small on {len(val_ds)} synthetic images...\n')

BATCH = 16
for start in tqdm(range(0, len(val_ds), BATCH), desc='Evaluating'):
    end = min(start + BATCH, len(val_ds))
    batch_samples = [val_ds[i] for i in range(start, end)]

    images = [s['image'].convert('RGB') for s in batch_samples]
    refs = [s['ogham_text'].replace('\u1680', ' ') for s in batch_samples]

    pixel_values = processor(images=images, return_tensors='pt').pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=64,
            num_beams=4,
            decoder_start_token_id=tokenizer.cls_token_id,
            eos_token_id=tokenizer.sep_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    all_preds.extend(preds)
    all_refs.extend(refs)

# Compute metrics
cer = compute_cer(all_preds, all_refs)
exact = sum(1 for p, r in zip(all_preds, all_refs) if p.strip() == r.strip()) / len(all_preds)

print(f'\n{"="*60}')
print(f'UNFINE-TUNED BASELINE RESULTS')
print(f'{"="*60}')
print(f'  Samples: {len(all_preds)}')
print(f'  Mean CER: {cer:.2%}')
print(f'  Exact Match: {exact:.1%}')
print(f'{"="*60}')

# Show sample predictions
print(f'\nSample predictions:')
for i in range(min(10, len(all_preds))):
    match = 'YES' if all_preds[i].strip() == all_refs[i].strip() else ''
    print(f"  ref='{all_refs[i][:30]}' pred='{all_preds[i][:30]}' {match}")

---
## 5. Compare with Fine-tuned

In [ ]:
print('\n' + '=' * 65)
print('FINE-TUNING IMPACT')
print('=' * 65)
print(f"{'Model':<30} {'CER':>10} {'Exact':>10}")
print('-' * 65)
print(f"{'TrOCR-small UNFINE-TUNED':<30} {cer:>9.2%} {exact:>9.1%}")
print(f"{'TrOCR-small frozen (20 ep)':<30} {'0.14%':>10} {'99.5%':>10}")
print(f"{'TrOCR-small unfrozen (20 ep)':<30} {'0.06%':>10} {'99.8%':>10}")
print('-' * 65)
improvement = cer / 0.0006 if cer > 0 else float('inf')
print(f'\nFine-tuning improvement: {improvement:.0f}x CER reduction')
print(f'From {cer:.2%} -> 0.06% in 20 epochs of training.')

In [ ]:
# Save results
import json

results = {
    'model': 'TrOCR-small-stage1 (unfine-tuned)',
    'dataset': HF_DATASET,
    'split': 'validation',
    'n_samples': len(all_preds),
    'mean_cer': cer,
    'exact_match': exact,
    'sample_predictions': [
        {'reference': all_refs[i], 'prediction': all_preds[i]}
        for i in range(min(20, len(all_preds)))
    ],
}

out_path = f'{DRIVE_ROOT}/experiments/unfinetuned_baseline.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'Results saved to: {out_path}')